In [1]:
# Cell 1: Import thư viện
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Cell 2: Đọc dữ liệu sạch
df = pd.read_csv("../data/processed/telco_churn_clean.csv")

print("Kích thước dữ liệu:", df.shape)
display(df.head())

Kích thước dữ liệu: (7032, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Cell 3: Kiểm tra thông tin dữ liệu
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   object 
 3   Dependents        7032 non-null   object 
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   object 
 6   MultipleLines     7032 non-null   object 
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   object 
 9   OnlineBackup      7032 non-null   object 
 10  DeviceProtection  7032 non-null   object 
 11  TechSupport       7032 non-null   object 
 12  StreamingTV       7032 non-null   object 
 13  StreamingMovies   7032 non-null   object 
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  7032 non-null   object 
 16  PaymentMethod     7032 non-null   object 


In [4]:
# Cell 4: Mã hóa biến mục tiêu
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print(df["Churn"].value_counts())
display(df.head())

Churn
0    5163
1    1869
Name: count, dtype: int64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [5]:
# Cell 5: Tách X và y
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)

Kích thước X: (7032, 19)
Kích thước y: (7032,)


In [6]:
# Cell 6: Xác định nhóm biến số và biến phân loại
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]

categorical_features = [col for col in X.columns if col not in numerical_features]

print("Biến số:", numerical_features)
print("Biến phân loại:", categorical_features)

Biến số: ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
Biến phân loại: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [7]:
# Cell 7: Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5625, 19)
X_test : (1407, 19)
y_train: (5625,)
y_test : (1407,)


In [8]:
# Cell 8: Preprocessor có scale cho model cần chuẩn hóa
numeric_transformer_scaled = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor_scaled = ColumnTransformer(transformers=[
    ("num", numeric_transformer_scaled, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

In [9]:
# Cell 9: Tiền xử lý dữ liệu cho model cần scale
X_train_processed = preprocessor_scaled.fit_transform(X_train)
X_test_processed = preprocessor_scaled.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape :", X_test_processed.shape)

X_train_processed shape: (5625, 30)
X_test_processed shape : (1407, 30)


In [10]:
# Cell 10: Lấy tên cột sau preprocessing
encoded_cat_features = preprocessor_scaled.named_transformers_["cat"] \
    .named_steps["onehot"] \
    .get_feature_names_out(categorical_features)

all_features = numerical_features + list(encoded_cat_features)

print("Tổng số feature sau encoding:", len(all_features))
print(all_features[:20])  # xem thử 20 feature đầu

Tổng số feature sau encoding: 30
['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes']


In [11]:
# Cell 11: Chuyển X_train_processed thành DataFrame
X_train_df = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed,
    columns=all_features,
    index=X_train.index
)

display(X_train_df.head())
print("Kích thước sau encoding:", X_train_df.shape)

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
1408,1.321816,0.981556,1.659900,-0.439319,1.0,1.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6992,-0.267410,-0.971546,-0.562252,-0.439319,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3349,1.444064,0.837066,1.756104,-0.439319,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4486,-1.204646,0.641092,-0.908326,-0.439319,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3535,0.669826,-0.808787,-0.101561,-0.439319,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Kích thước sau encoding: (5625, 30)


In [12]:
# Cell 12: Preprocessor không scale cho tree models
numeric_transformer_no_scale = Pipeline(steps=[
    ("passthrough", "passthrough")
])

preprocessor_no_scale = ColumnTransformer(transformers=[
    ("num", numeric_transformer_no_scale, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
# Cell 13: Transform cho tree-based models

X_train_tree = preprocessor_no_scale.fit_transform(X_train)
X_test_tree = preprocessor_no_scale.transform(X_test)

print("X_train_tree shape:", X_train_tree.shape)
print("X_test_tree shape :", X_test_tree.shape)

X_train_tree shape: (5625, 30)
X_test_tree shape : (1407, 30)


In [14]:
# Cell 14: Lưu dữ liệu train đã encode thành DataFrame
X_train_tree_df = pd.DataFrame(
    X_train_tree.toarray() if hasattr(X_train_tree, "toarray") else X_train_tree,
    columns=all_features,
    index=X_train.index
)

X_test_tree_df = pd.DataFrame(
    X_test_tree.toarray() if hasattr(X_test_tree, "toarray") else X_test_tree,
    columns=all_features,
    index=X_test.index
)

X_train_tree_df.to_csv("../data/processed/X_train_encoded.csv", index=False)
X_test_tree_df.to_csv("../data/processed/X_test_encoded.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Đã lưu các file processed vào thư mục data/processed/")

Đã lưu các file processed vào thư mục data/processed/
